In [4]:
# чтение данных из базы
import pandas as pd
import os, psycopg
from dotenv import load_dotenv
load_dotenv()

TABLE_NAME = "users_churn"# таблица с данными


connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

df[:2]

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,2043,7361-YPXFS,2017-10-01,NaT,Month-to-month,No,Bank transfer (automatic),64.45,1867.6,DSL,...,Yes,Yes,No,No,Female,1,No,No,Yes,0
1,2044,6557-BZXLQ,2018-10-01,NaT,Month-to-month,No,Electronic check,69.65,1043.3,Fiber optic,...,No,No,No,No,Male,1,No,No,No,0


In [6]:
# разделение выборки
from sklearn.model_selection import train_test_split

features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = "customer_id"  # Замените на подходящий столбец
stratify_column = "stratify_column"  # Зафиксируйте целевую переменную для стратификации
test_size = 0.2  # Размер тестовой выборки

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target], test_size=test_size, shuffle=False
)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")


Размер выборки для обучения: (5634, 3)
Размер выборки для теста: (1409, 3)


In [12]:
# подбор параметров
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV


loss_function = "Logloss"
task_type = 'CPU'
random_seed = 0
iterations = 300
verbose = False

params = {
    'depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.1, 0.5],
    'l2_leaf_reg': [1, 3, 5, 10, 15],
}

model = CatBoostClassifier(
    **params,
    loss_function=loss_function,
    verbose=verbose,
    iterations=iterations,
    task_type=task_type,
    random_seed=random_seed
)

cv = GridSearchCV(param_grid=params, estimator=model, cv=2, n_jobs=-1, verbose=2)

# Обучение модели
clf = cv.fit(X_train, y_train)

Fitting 2 folds for each of 75 candidates, totalling 150 fits
[CV] END .........depth=3, l2_leaf_reg=1, learning_rate=0.01; total time=   1.4s
[CV] END ..........depth=3, l2_leaf_reg=1, learning_rate=0.1; total time=   1.0s
[CV] END ..........depth=3, l2_leaf_reg=1, learning_rate=0.5; total time=   1.0s
[CV] END .........depth=3, l2_leaf_reg=3, learning_rate=0.01; total time=   0.8s
[CV] END ..........depth=3, l2_leaf_reg=3, learning_rate=0.1; total time=   0.9s
[CV] END .........depth=3, l2_leaf_reg=5, learning_rate=0.01; total time=   0.8s
[CV] END ..........depth=3, l2_leaf_reg=5, learning_rate=0.1; total time=   0.6s
[CV] END ..........depth=3, l2_leaf_reg=5, learning_rate=0.5; total time=   1.1s
[CV] END ..........depth=3, l2_leaf_reg=5, learning_rate=0.5; total time=   0.7s
[CV] END ........depth=3, l2_leaf_reg=10, learning_rate=0.01; total time=   0.6s
[CV] END .........depth=3, l2_leaf_reg=10, learning_rate=0.1; total time=   1.0s
[CV] END .........depth=3, l2_leaf_reg=10, lear

In [14]:
# проверка модели с лучшими параметрами

cv_results = pd.DataFrame(clf.cv_results_)

best_params = clf.best_params_

model_best = CatBoostClassifier(
    **best_params,
    loss_function=loss_function,
    verbose=verbose,
    iterations=iterations,
    task_type=task_type,
    random_seed=random_seed
)

model_best.fit(X_train, y_train)

prediction = model_best.predict(X_test)
probas = model_best.predict_proba(X_test)[:, 1]

In [18]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss

metrics = {}

_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

# сохранение метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

# дополнительные метрики из результатов кросс-валидации
metrics['mean_fit_time'] = cv_results['mean_fit_time'].mean()
metrics['mean_test_score'] = cv_results['mean_test_score'].mean()
metrics['std_test_score'] = cv_results['std_test_score'].mean()
metrics['best_score'] = clf.best_score_
metrics['std_fit_time'] = cv_results['std_fit_time'].mean()


0.5226480836236934

In [28]:
# логируем результат
import mlflow

EXPERIMENT_NAME = "churn"
RUN_NAME = 'model_grid_search' # ваш код здесь
REGISTRY_MODEL_NAME = "churn_model_arvas"

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = "YCAJE3Nlz8iDILW5VTYM1ihQB"
os.environ["AWS_SECRET_ACCESS_KEY"] = "YCPjvS7uwhvJpUj3bKm8X-IX4QAwBIVsvX61IL44"

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_registry_uri("http://localhost:5000")

# настройки для логирования в MLFlow
pip_requirements = 'requirements.txt'
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id


with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_params(best_params)
    mlflow.log_metrics(metrics)
    cv_info = mlflow.sklearn.log_model(cv, artifact_path='cv')
    model_info = mlflow.catboost.log_model(
        cb_model=model_best,
        artifact_path='models',
        input_example=input_example,
        signature=signature,
        registered_model_name=REGISTRY_MODEL_NAME,
        pip_requirements=pip_requirements
    )


/home/mle-user/mle-projects/mle-mlflow/.venv_notebook/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Registered model 'churn_model_arvas' already exists. Creating a new version of this model...
2025/09/15 06:25:17 INFO mlflow.tracking._model_r